# Семинар 1. Gymnasium, политики руками, первые эксперименты

План:

1. Интерфейс Gymnasium на примере `FrozenLake-v1`
2. Политика как таблица: пишем маршрут по льду руками
3. Что ломается, когда лёд скользкий
4. CartPole: улучшаем эвристику из лекции
5. Что дальше: PyTorch-разминка и домашнее задание

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

rng = np.random.default_rng(42)

## 1. Интерфейс Gymnasium

Любая среда в Gymnasium следует единому интерфейсу:

* `env.reset(seed=...)` -> `(observation, info)` — сбросить среду в начальное состояние
* `env.step(action)` -> `(observation, reward, terminated, truncated, info)` — сделать шаг
* `env.observation_space`, `env.action_space` — описание пространств состояний/действий

`terminated` — эпизод закончился естественным образом (например, дошли до цели или упали),
`truncated` — эпизод прерван искусственно (например, по лимиту шагов).

Начнём с самой простой среды: **FrozenLake**. Агент ходит по замёрзшему озеру 4×4 от старта `S`
к цели `G`, в клетках `H` — проруби, попал в прорубь — эпизод окончен с наградой 0.
Награда +1 только за достижение цели.

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=False)
print("observation_space:", env.observation_space)
print("action_space:", env.action_space)

obs, info = env.reset(seed=0)
print("начальное состояние:", obs)

for step in range(5):
    action = env.action_space.sample()  # случайное действие
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"step={step} action={action} obs={obs} reward={reward} terminated={terminated}")
    if terminated or truncated:
        obs, info = env.reset()

env.close()

Состояние здесь — просто номер клетки от 0 до 15 (нумерация по строкам, слева направо).
Действия: `0` — влево, `1` — вниз, `2` — вправо, `3` — вверх. Посмотрим на карту.

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array")
obs, _ = env.reset(seed=0)
desc = env.unwrapped.desc.astype(str)
print("карта озера (S — старт, F — лёд, H — прорубь, G — цель):")
for row in desc:
    print("  ", " ".join(row))
print()
print("номера состояний:")
print(np.arange(16).reshape(4, 4))

plt.imshow(env.render())
plt.axis("off")
plt.title("FrozenLake-v1: агент в состоянии 0")
plt.show()
env.close()

## 2. Политика как таблица

В лекции политика — правило «в такой ситуации делаю так». Когда состояний всего 16, это правило
можно записать **таблицей**: для каждой клетки одно действие. Это самая простая форма политики,
и именно её мы будем *обучать* на неделях 2–3. Сегодня напишем её руками.

Задание: заполните словарь `policy` так, чтобы агент дошёл от клетки 0 до клетки 15, не наступая
в проруби (клетки 5, 7, 11, 12). Для клеток, куда агент по вашему маршруту не попадает, действие
можно не задавать (по умолчанию будет `0`).

In [ ]:
LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3

policy = {
    0: DOWN,
    4: DOWN,
    8: RIGHT,
    9: DOWN,
    13: RIGHT,
    14: RIGHT,
}


def table_policy(obs):
    return policy.get(int(obs), LEFT)


def run_episode(env, policy_fn, seed=None, verbose=False):
    obs, _ = env.reset(seed=seed)
    total, t = 0.0, 0
    while True:
        action = policy_fn(obs)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        if verbose:
            print(f"t={t}: состояние {obs} -> действие {action} -> состояние {next_obs}, награда {reward}")
        total += reward
        obs = next_obs
        t += 1
        if terminated or truncated:
            return total


env = gym.make("FrozenLake-v1", is_slippery=False)
G = run_episode(env, table_policy, seed=0, verbose=True)
print("суммарная награда за эпизод:", G)
env.close()

## 3. Скользкий лёд

Настоящая FrozenLake по умолчанию **скользкая**: агент идёт туда, куда хотел, только с вероятностью 1/3,
а с вероятностью 2/3 его сносит в одну из перпендикулярных сторон. Это первый пример **случайной среды**:
одно и то же действие в одном и том же состоянии приводит к разным исходам.

Проверим ту же таблицу на скользком льду. Одного эпизода теперь мало: нужно много запусков и доля успехов.

In [ ]:
def success_rate(env, policy_fn, n_episodes=500):
    wins = 0
    for ep in range(n_episodes):
        wins += run_episode(env, policy_fn, seed=ep) > 0
    return wins / n_episodes


def random_policy(obs):
    return int(rng.integers(4))


for slippery in [False, True]:
    env = gym.make("FrozenLake-v1", is_slippery=slippery)
    print(f"is_slippery={slippery!s:5}: таблица {success_rate(env, table_policy):.1%}, "
          f"случайная политика {success_rate(env, random_policy):.1%}")
    env.close()

Маршрут, идеальный на гладком льду, на скользком доходит до цели лишь в небольшой доле случаев:
его сносит в проруби, а таблица не знает, что делать в клетках, куда агент «не собирался».

Попробуйте вживую:

* Дополните таблицу действиями для всех 16 клеток (для прорубей и цели — что угодно) и посмотрите, растёт ли доля успехов.
* Есть ли маршрут «в обход» прорубей, который надёжнее? Подсказка: на скользком льду бывает выгодно
  «идти в стену», потому что стена не пускает, а снос работает в нужную сторону.
* На неделе 2 мы получим для этой среды **оптимальную** таблицу автоматически и сравним с вашей.

In [ ]:
# Место для экспериментов: ваша улучшенная таблица для скользкого льда
policy_slippery = dict(policy)
# policy_slippery[1] = ...
# policy_slippery[2] = ...

env = gym.make("FrozenLake-v1", is_slippery=True)
print(f"улучшенная таблица на скользком льду: {success_rate(env, lambda o: policy_slippery.get(int(o), LEFT)):.1%}")
env.close()

## 4. CartPole: улучшаем эвристику из лекции

Вернёмся к тележке с шестом. Наблюдение — 4 числа: положение тележки $x$, её скорость $\dot x$,
угол шеста $\theta$ и угловая скорость $\dot\theta$. Действие: `0` — толкнуть влево, `1` — вправо.

Эвристика из лекции смотрела только на $\dot\theta$ и держала шест около 200 шагов из 500 возможных.
Задача: **придумать правило, которое держит шест дольше**, глядя на все четыре числа.
Идеи для обсуждения:

* учитывать не только скорость, но и сам угол $\theta$ (шест уже наклонён, хотя пока не падает);
* следить за тележкой: если она уезжает к краю, чуть «подталкивать» её обратно;
* взвешенная сумма всех четырёх чисел с порогом: $a = [w \cdot obs > 0]$. Подберите веса руками.

In [ ]:
def run_episodes(env_id, policy_fn, n_episodes=30, seed=0):
    env = gym.make(env_id)
    returns = []
    for ep in range(n_episodes):
        returns.append(run_episode(env, policy_fn, seed=seed + ep))
    env.close()
    return np.array(returns)


def cartpole_random(obs):
    return int(rng.integers(2))


def cartpole_lecture(obs):
    x, x_dot, theta, theta_dot = obs
    return int(theta_dot > 0)


def cartpole_improved(obs):
    x, x_dot, theta, theta_dot = obs
    # TODO: ваша версия. Пока здесь копия эвристики из лекции: добавьте угол, потом положение тележки
    return int(theta_dot > 0)


results = {}
for name, fn in [("случайная", cartpole_random), ("из лекции", cartpole_lecture), ("улучшенная", cartpole_improved)]:
    results[name] = run_episodes("CartPole-v1", fn)
    print(f"{name:11s}: средняя суммарная награда {results[name].mean():6.1f} ± {results[name].std():5.1f} (максимум 500)")

In [ ]:
plt.figure(figsize=(8, 3.5))
for name, G in results.items():
    plt.hist(G, bins=np.linspace(0, 500, 26), alpha=0.6, label=name)
plt.xlabel("суммарная награда за эпизод")
plt.ylabel("число эпизодов")
plt.title("CartPole: разброс результатов трёх политик")
plt.legend()
plt.show()

Обратите внимание: даже у хорошей ручной политики результаты от эпизода к эпизоду сильно разные.
Начальное состояние случайное, и одна и та же политика может продержаться и 100, и 500 шагов.
Поэтому в RL **никогда не судят по одному эпизоду**: всегда среднее по многим запускам, и желательно с разбросом.

## Что дальше

* **Мини-семинар по PyTorch** (`pytorch_intro.ipynb`): тензоры, autograd, `nn.Module`, цикл обучения.
  В конце — первый «агент на нейросети»: обучим сеть повторять вашу улучшенную эвристику для CartPole.
* **Домашнее задание** (`../homework/homework.ipynb`): ручные политики для CartPole и FrozenLake с автопроверкой,
  формулировка трёх задач из жизни на языке RL, мини-эксперимент с исследованием.
* **Неделя 2**: многорукие бандиты (задача с кафе из лекции, но с настоящими алгоритмами), MDP и уравнения Беллмана;
  для FrozenLake найдём оптимальную таблицу автоматически.